## Patient Journey - T2D
#### Data source used

* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 
* finerenone_intermediate.diabetes_product
* finerenone_intermediate.diabetes_product_injection_mapping

#### Steps:
1. Query TYPE 2 DIABETES diagnosis claims data after 2016-01-01, RX claim data with T2D product flag
2. Combine these two data sources to generate T2D flag for each patient
3. Save table to databricks

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run "./00_cohort_config"

### Get Data from Scope

# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

OFFSET_DAY      = 7
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


##   Identify T2D patients from Diagnosis and RX data

In [0]:
# changed hist table

query = """
    SELECT  DISTINCT PATIENT_ID
            FROM (
          SELECT DX.CLAIM_ID, DX.PATIENT_ID,DX.DIAGNOSIS_CODE,DX.PROVIDER_RENDERING_ID
          FROM PHCDW.PHCDW_DISCOVERY.STG_IQV_LAAD_DX DX
          INNER JOIN 
          ( SELECT  CLAIM_ID, MAX(LAST_DATE) AS LAST_DATE
                    FROM PHCDW.PHCDW_DISCOVERY.STG_IQV_LAAD_DX
                    GROUP BY claim_id) MAX_DX
          ON (DX.CLAIM_ID = MAX_DX.CLAIM_ID AND DX.LAST_DATE = MAX_DX.LAST_DATE)
          WHERE SERVICE_DATE <= '2018-05-31'

          UNION ALL

          SELECT CLAIM_ID, TO_VARCHAR(PATIENT_ID) AS PATIENT_ID, DIAGNOSIS_CODE, TO_VARCHAR(PROVIDER_RENDERING_ID) AS PROVIDER_RENDERING_ID
          FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX

          ) DX    
                  
          Join          
            
          (SELECT DISTINCT DIAGNOSIS_CODE,DIAGNOSIS_DESCRIPTION 
          FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
                            ) CD  
          ON DX.DIAGNOSIS_CODE=CD.DIAGNOSIS_CODE
          WHERE UPPER(DIAGNOSIS_DESCRIPTION) LIKE '%TYPE 2 DIABETES%'
            AND PROVIDER_RENDERING_ID IS NOT NULL
   """
#df_dx_patient = (spark.read.format(SNOWFLAKE_SOURCE_NAME).options(**sfOptions).option("query", query).load())
df_dx_patient = Get_Data_Snowflakes(query)

In [0]:
df_dx_patient.count()

54175121

In [0]:
df_product_mapping = spark.sql("select * from kerendia_pcp.diabetes_product")
df_injection_mapping = spark.sql("select * from kerendia_pcp.diabetes_product_injection_mapping")

In [0]:
query = """
        SELECT 
        DISTINCT NDC_CD, PRODUCT_NAME,GENERIC_NAME, PRODUCT_STRENGTH, PRODUCT_FORM
        FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT
   """
df_product = Get_Data_Snowflakes(query)

In [0]:
df_product.display()

NDC_CD,PRODUCT_NAME,GENERIC_NAME,PRODUCT_STRENGTH,PRODUCT_FORM
00009344901,GLYNASE,GLYBURIDE MICRONIZED,6 MG,TABLET
00047044024,FUROSEMIDE,FUROSEMIDE,20 MG,TABLET
00003422216,KOMBIGLYZE XR,SAXAGLIPTIN-METFORMIN HCL,2.5-1000 MG/24HR,TABLET SR 24 HR
00049017403,GLUCOTROL XL,GLIPIZIDE,5 MG/24HR,TABLET SR 24 HR
00078046205,LOPRESSOR HCT,METOPROLOL & HYDROCHLOROTHIAZIDE,100-50 MG,TABLET
00005310223,PROPRANOLOL HCL,PROPRANOLOL HCL,60 MG,TABLET
00085024407,NORMODYNE,LABETALOL HCL,100 MG,TABLET
00002821502,HUMULIN R,null,100U/ML,null
00002871759,HUMULIN 70/30,INSULIN NPH ISOPHANE & REG (HUMAN),100 UNIT/ML,SUSPENSION
00024585030,AVAPRO,IRBESARTAN,75 MG,TABLET


In [0]:
df_product_mapping.display()

PRODUCT_NAME,CONFLUENCE,LOGIC
V-GO 20,null,null
V-GO 30,null,null
V-GO 40,null,null
OMNIPOD 5 PACK,null,null
OMNIPOD DASH 5 PACK,null,null
HUMULIN 70/30,insulin (human),2
HUMULIN 70/30 KWIKPEN,insulin (human),2
HUMULIN 70/30 PEN,insulin (human),2
HUMULIN N,insulin (human),2
HUMULIN N KWIKPEN,insulin (human),2


In [0]:
df_injection_mapping.display()

PRODUCT_FORM,INJECTION
CAPSULE,No
CAPSULE ER 24 HOUR SPRINKLE,No
CAPSULE EXTENDED RELEASE,No
CAPSULE SR 12 HR,No
CAPSULE SR 24 HR,No
ELIXIR,No
EMULSION,No
GEL (JELLY),No
LIQUID,No
MISCELLANEOUS,No


In [0]:

from pyspark.sql.functions import col, when, trim

df_product_flag = (
  df_product
  .join(df_product_mapping, how="inner", on="PRODUCT_NAME")
  .join(df_injection_mapping, how="left", on="PRODUCT_FORM")
  .filter(
      col("PRODUCT_NAME").isNotNull() &
      col("PRODUCT_FORM").isNotNull() &
      (trim(col("PRODUCT_FORM")) != "") &
      col("LOGIC").isNotNull() &
      (~col("LOGIC").cast("string").isin("null", "")) & 
      col("INJECTION").isNotNull() &
      (~col("INJECTION").isin("null", ""))
  )
  .withColumn(
      "DIABETIC_FLAG",
      when(col("LOGIC").cast("int") == 1, 1).otherwise(
          when(col("INJECTION") == "No", 1).otherwise(0)
      )
  )
)



In [0]:
df_product_flag.display()

PRODUCT_FORM,PRODUCT_NAME,NDC_CD,GENERIC_NAME,PRODUCT_STRENGTH,CONFLUENCE,LOGIC,INJECTION,DIABETIC_FLAG
TABLET SR 24 HR,KOMBIGLYZE XR,00003422216,SAXAGLIPTIN-METFORMIN HCL,2.5-1000 MG/24HR,"metformin, saxagliptin and dapagliflozin",1,No,1
SUSPENSION,HUMULIN 70/30,00002871759,INSULIN NPH ISOPHANE & REG (HUMAN),100 UNIT/ML,insulin (human),2,Yes,0
TABLET,AVANDIA,00029315913,ROSIGLITAZONE MALEATE,4 MG,rosiglitazone,1,No,1
SOLUTION,HUMULIN R U-500 (CONCENTR,00002850101,INSULIN REGULAR (HUMAN),500 UNIT/ML,insulin (human),2,Yes,0
SOLUTION PEN-INJECTOR,HUMALOG KWIKPEN,00002771201,INSULIN LISPRO,200 UNIT/ML,insulin lispro,2,Yes,0
TABLET SR 24 HR,GLUCOTROL XL,00049017807,GLIPIZIDE,10 MG/24HR,glipizide,1,No,1
TABLET SR 24 HR,GLUCOTROL XL,00049155073,GLIPIZIDE,5 MG/24HR,glipizide,1,No,1
TABLET,GLYBURIDE/METFORMIN HYDRO,00093571001,GLYBURIDE-METFORMIN,1.25-250 MG,glibenclamide,1,No,1
TABLET,JANUVIA,00006022101,SITAGLIPTIN PHOSPHATE,25 MG,sitagliptin,1,No,1
TABLET SR 24 HR,XIGDUO XR,00310625030,DAPAGLIFLOZIN-METFORMIN HCL,5-500 MG/24HR,metformin and dapagliflozin,1,No,1


In [0]:
# UPDATE - changed rx_hist table

query = """
        SELECT  DISTINCT PATIENT_ID, NDC_CD
        FROM 
        (
        SELECT DISTINCT CLAIM_ID, TO_VARCHAR(PATIENT_ID) AS PATIENT_ID, NDC_CD, SVC_DT
        FROM    (
                SELECT PATIENT_ID, CLAIM_ID, SVC_DT, DAYS_SUPPLY, QUANTITY, REFILL_CODE, REJECT_CODE, PROVIDER_ID, NDC_CD, PAYER_PLAN_ID 
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX where PATIENT_ID is not null
                ) RX
        ) where PATIENT_ID is not null
   """

df_rx = Get_Data_Snowflakes(query)

In [0]:
df_rx.count()
df_rx.limit(100).display()

PATIENT_ID,NDC_CD
1776734783,68180072003
1575079342,68180051303
1721547353,00597015390
1719183356,16729022601
382374167,65862047090
638925853,43547042309
14007513905,00002147180
988424953,69097012715
2565282528,52817025010
139902289,31722070010


In [0]:
df_rx_patient = (
  df_rx
  .join(df_product_flag, how="inner", on="NDC_CD")
  .filter("DIABETIC_FLAG = 1")
  .filter(col("PATIENT_ID").isNotNull())  # <-- filters out nulls
  .select("PATIENT_ID")
  .distinct()
)

In [0]:
df_rx_patient.count()

51659891

In [0]:
df_t2d_patient = (
  df_dx_patient.union(df_rx_patient)
  .dropDuplicates()
  .withColumn("DIABETIC_FLAG", F.lit(1))
)

In [0]:
df_t2d_patient.display()

PATIENT_ID,DIABETIC_FLAG
597794894,1
956375176,1
841071045,1
168793401,1
769121327,1
1815543997,1
1029929938,1
789994962,1
72585057,1
3486072123,1


In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_t2d_patient.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_t2d_patient_validation")